In [1]:
from google.colab import drive
drive.mount("/content/drive")

import os
import torch

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["PYTHONHASHSEED"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

assert torch.cuda.is_available(), "Enable a GPU runtime first."

gpu_name = torch.cuda.get_device_name(0)
gpu_gib = torch.cuda.get_device_properties(0).total_memory / 2**30

print("GPU:", gpu_name)
print(f"VRAM: {gpu_gib:.1f} GiB")
print("CUDA:", torch.version.cuda)

assert "A100" in gpu_name, f"Expected A100, received {gpu_name}"

Mounted at /content/drive
GPU: NVIDIA A100-SXM4-40GB
VRAM: 39.5 GiB
CUDA: 12.8


In [2]:
from pathlib import Path
import os
import subprocess

REPO = Path("/content/MailoHLS")
COMMIT = input("Paste the post-fix MailoHLS commit SHA: ").strip()

assert len(COMMIT) == 40

if not REPO.exists():
    subprocess.run(
        [
            "git", "clone",
            "https://github.com/ElenaVouvali/MailoHLS.git",
            str(REPO),
        ],
        check=True,
    )

subprocess.run(["git", "fetch", "--all"], cwd=REPO, check=True)
subprocess.run(["git", "checkout", "--detach", COMMIT], cwd=REPO, check=True)

actual = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    cwd=REPO,
    text=True,
).strip()

assert actual == COMMIT
os.chdir(REPO)
print("Checked out:", actual)

Paste the post-fix MailoHLS commit SHA: 082ef85759e50c683bee780c6a51c5e93b7e809d
Checked out: 082ef85759e50c683bee780c6a51c5e93b7e809d


In [3]:
import subprocess
import sys

packages = [
    "transformers==4.51.0",
    "peft==0.18.1",
    "accelerate==1.13.0",
    "bitsandbytes==0.49.2",
    "einops==0.8.2",
    "einops-exts==0.0.4",
    "safetensors==0.7.0",
    "sentencepiece",
    "pytest",
]

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", *packages],
    check=True,
)

import inspect
import transformers
import peft
import accelerate
import bitsandbytes
from peft import LoraConfig

assert "trainable_token_indices" in inspect.signature(
    LoraConfig
).parameters

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
print("accelerate:", accelerate.__version__)
print("bitsandbytes:", bitsandbytes.__version__)

torch: 2.11.0+cu128
transformers: 4.51.0
peft: 0.18.1
accelerate: 1.13.0
bitsandbytes: 0.49.2


In [4]:
from pathlib import Path
from collections import defaultdict
import hashlib
import json
import re

DRIVE_DATASET_ROOT = Path(
    "/content/drive/MyDrive/MailoHLS/LLM_data"
)

DATASET_PATH = DRIVE_DATASET_ROOT / "mailohls_sft.jsonl"
REGISTRY_PATH = DRIVE_DATASET_ROOT / "directive_domain_registry.json"

assert DATASET_PATH.is_file(), DATASET_PATH

ASSIGN_RE = re.compile(
    r"^(auto\{_[A-Z0-9]+(?:_[A-Z0-9]+)*_L\d+\})\s*=\s*(.+)$",
    re.IGNORECASE,
)


def normalize_kernel_name(name):
    # Matches train_SFT_xattn_new.py::normalize_kname()
    return re.sub(r"[-\s]+", "_", str(name).strip().lower())


def rhs_sort_key(value):
    value = str(value).strip()
    if re.fullmatch(r"-?\d+", value):
        return (0, int(value), "", value)
    return (1, 0, value.lower(), value)


domains = defaultdict(lambda: defaultdict(set))
raw_kernel_for_normalized = {}

num_rows = 0
num_assignments = 0

with DATASET_PATH.open("r", encoding="utf-8") as handle:
    for line_number, line in enumerate(handle, start=1):
        if not line.strip():
            continue

        row = json.loads(line)
        num_rows += 1

        raw_kernel = str(row["kernel_name"]).strip()
        kernel = normalize_kernel_name(raw_kernel)

        previous = raw_kernel_for_normalized.setdefault(kernel, raw_kernel)
        if previous != raw_kernel:
            raise ValueError(
                f"Kernel-name normalization collision: "
                f"{previous!r} and {raw_kernel!r} -> {kernel!r}"
            )

        target = str(row.get("target", "")).strip()
        if not target:
            raise ValueError(f"Missing target at JSONL line {line_number}")

        row_assignments = 0

        for target_line in target.splitlines():
            target_line = target_line.strip()
            if not target_line:
                continue

            match = ASSIGN_RE.fullmatch(target_line)
            if match is None:
                raise ValueError(
                    f"Malformed target assignment at JSONL line "
                    f"{line_number}: {target_line!r}"
                )

            lhs = match.group(1).upper()
            rhs = match.group(2).strip()

            if not rhs or rhs == "?":
                raise ValueError(
                    f"Invalid RHS at JSONL line {line_number}: "
                    f"{target_line!r}"
                )

            domains[kernel][lhs].add(rhs)
            row_assignments += 1
            num_assignments += 1

        if row_assignments == 0:
            raise ValueError(
                f"No directive assignments at JSONL line {line_number}"
            )

registry = {
    kernel: {
        lhs: sorted(values, key=rhs_sort_key)
        for lhs, values in sorted(sites.items())
    }
    for kernel, sites in sorted(domains.items())
}

payload = {
    "schema": "mailohls-directive-domain-registry-v1",
    "source_dataset": DATASET_PATH.name,
    "source_dataset_sha256": hashlib.sha256(
        DATASET_PATH.read_bytes()
    ).hexdigest(),
    "generation_policy": (
        "exact per-kernel/per-site RHS support from the complete "
        "pre-split MailoHLS dataset"
    ),
    "kernels": registry,
}

temporary_path = REGISTRY_PATH.with_suffix(".json.tmp")
temporary_path.write_text(
    json.dumps(payload, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)
temporary_path.replace(REGISTRY_PATH)

num_sites = sum(len(sites) for sites in registry.values())

print("Wrote:", REGISTRY_PATH)
print("Rows scanned:", num_rows)
print("Assignments scanned:", num_assignments)
print("Kernels:", len(registry))
print("Directive sites:", num_sites)
print(
    "Registry SHA256:",
    hashlib.sha256(REGISTRY_PATH.read_bytes()).hexdigest(),
)

Wrote: /content/drive/MyDrive/MailoHLS/LLM_data/directive_domain_registry.json
Rows scanned: 150026
Assignments scanned: 4645682
Kernels: 55
Directive sites: 1299
Registry SHA256: 782246f880cc30e93c6499b3440fccdc22f9ce51f7b8d01dc9b32d853d5279ac


In [5]:
from pathlib import Path
import hashlib
import json
import shutil
from collections import Counter

DRIVE_DATASET_ROOT = Path(
    "/content/drive/MyDrive/MailoHLS/LLM_data"
)
LOCAL_DATASET_ROOT = REPO / "artifacts/llm"

required_files = [
    "mailohls_sft.jsonl",
    "mailohls_sft.sources.json",
    "mailohls_sft.manifest.json",
    "directive_domain_registry.json",
]

LOCAL_DATASET_ROOT.mkdir(parents=True, exist_ok=True)

for filename in required_files:
    source = DRIVE_DATASET_ROOT / filename
    destination = LOCAL_DATASET_ROOT / filename

    assert source.is_file(), f"Missing required dataset file: {source}"
    shutil.copy2(source, destination)
    print("Copied:", destination)

DATASET = LOCAL_DATASET_ROOT / "mailohls_sft.jsonl"
DIRECTIVE_DOMAIN_REGISTRY = LOCAL_DATASET_ROOT / "directive_domain_registry.json"

digest = hashlib.sha256(DATASET.read_bytes()).hexdigest()

rows = []
with DATASET.open("r", encoding="utf-8") as handle:
    for line_number, line in enumerate(handle, start=1):
        if not line.strip():
            continue
        row = json.loads(line)
        assert "kernel_name" in row, line_number
        rows.append(row)

print("dataset:", DATASET)
print("sha256:", digest)
print("rows:", len(rows))
print("devices:", Counter(str(r.get("device")) for r in rows))
print("kernels:", len({r["kernel_name"] for r in rows}))

assert rows



Copied: /content/MailoHLS/artifacts/llm/mailohls_sft.jsonl
Copied: /content/MailoHLS/artifacts/llm/mailohls_sft.sources.json
Copied: /content/MailoHLS/artifacts/llm/mailohls_sft.manifest.json
Copied: /content/MailoHLS/artifacts/llm/directive_domain_registry.json
dataset: /content/MailoHLS/artifacts/llm/mailohls_sft.jsonl
sha256: 0a235f79abd4b065c0cff0e943ab91ca20408cfe69358b3b61051a98216d6a14
rows: 150026
devices: Counter({'xczu7ev-ffvc1156-2-e': 78571, 'xcu200-fsgd2104-2-e': 71455})
kernels: 55


In [6]:
import json
import sys

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from LLM_branch.train.train_SFT_xattn_new import (
    extract_ordered_lhs_plan,
    load_directive_domain_registry,
    normalize_kname,
)

registry = load_directive_domain_registry(
    str(DIRECTIVE_DOMAIN_REGISTRY)
)

sources_payload = json.loads(
    (
        LOCAL_DATASET_ROOT
        / "mailohls_sft.sources.json"
    ).read_text(encoding="utf-8")
)

templates = sources_payload["templates"]
problems = []

for raw_kernel, source_text in templates.items():
    kernel = normalize_kname(raw_kernel)

    expected_sites = {
        lhs.upper()
        for _, lhs in extract_ordered_lhs_plan(source_text)
    }
    registered_sites = set(registry.get(kernel, {}))

    missing = sorted(expected_sites - registered_sites)
    extra = sorted(registered_sites - expected_sites)

    if missing or extra:
        problems.append({
            "kernel": raw_kernel,
            "missing": missing,
            "extra": extra,
        })

assert not problems, problems[:5]

print("Registry validation passed")
print("Kernels:", len(registry))
print(
    "Sites:",
    sum(len(sites) for sites in registry.values()),
)

Registry validation passed
Kernels: 55
Sites: 1299


In [7]:
import subprocess
import sys

# 1. Syntax check the exact trainer + inference scripts
subprocess.run(
    [
        sys.executable, "-m", "py_compile",
        "LLM_branch/train/train_SFT_xattn_new.py",
        "LLM_branch/inference/eval_stage1_stage2_stage3.py",
    ],
    cwd=REPO,
    check=True,
)

# 2. Run all critical Stage1/Stage2 regression tests
subprocess.run(
    [
        sys.executable, "-m", "pytest", "-q",
        "LLM_branch/tests/test_structural_xattn.py",
        "LLM_branch/tests/test_stage1_adapter_roundtrip.py",
        "LLM_branch/tests/test_training_contract.py",
        "LLM_branch/tests/test_inference_schema.py",
    ],
    cwd=REPO,
    check=True,
)

print("All MailoHLS preflight tests passed.")

All MailoHLS preflight tests passed.


In [8]:
from pathlib import Path
import os
import sys

LOCAL_RUN_ROOT = Path("/content/mailohls_runs")
LOCAL_RUN_ROOT.mkdir(parents=True, exist_ok=True)

PERSIST_ROOT = Path(
    "/content/drive/MyDrive/MailoHLS/Experiments"
)
PERSIST_ROOT.mkdir(parents=True, exist_ok=True)

OBJECTIVE = "PARETO_ADP"

GOAL_TAG = {
    "PARETO_LATENCY": "pareto_latency",
    "PARETO_ADP": "pareto_adp",
    "PARETO_AREA": "pareto_area",
}[OBJECTIVE]

RUN_ID = "s123_inference_fix"

SPLIT_JSON = (
    PERSIST_ROOT /
    "stage1_stage2_family_split_s123.json"
)

SMOKE_DIR = (
    LOCAL_RUN_ROOT /
    f"stage1_{GOAL_TAG}_{RUN_ID}_smoke"
)

FULL_DIR = (
    PERSIST_ROOT /
    f"stage1_{GOAL_TAG}_{RUN_ID}_full"
)

STAGE2_SMOKE_DIR = (
    PERSIST_ROOT /
    f"stage2_{GOAL_TAG}_{RUN_ID}_smoke"
)

MEMORY_DIR = (
    REPO /
    "artifacts/stage2_inputs/"
    "memory_gnn_stage2_prod_s123"
)

assert MEMORY_DIR.is_dir(), MEMORY_DIR

if gpu_gib >= 70:
    batch_size = 2
    grad_accum = 4
else:
    batch_size = 1
    grad_accum = 8

print("batch_size:", batch_size)
print("grad_accum:", grad_accum)
print("effective batch:", batch_size * grad_accum)

ENV = os.environ.copy()
ENV["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
ENV["PYTHONHASHSEED"] = "0"
ENV["TOKENIZERS_PARALLELISM"] = "false"
ENV["PYTHONUNBUFFERED"] = "1"

ENV["PYTHONPATH"] = (
    f"{REPO}:"
    + ENV.get("PYTHONPATH", "")
)

from huggingface_hub import model_info
from transformers import AutoConfig, AutoTokenizer

MODEL_ID = "deepseek-ai/deepseek-coder-6.7b-base"
MODEL_REVISION = model_info(MODEL_ID).sha

config = AutoConfig.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
)

assert config.max_position_embeddings >= 7168

print("Model:", MODEL_ID)
print("Revision:", MODEL_REVISION)

COMMON_BASE = [
    sys.executable, "-u",
    "LLM_branch/train/train_SFT_xattn_new.py",

    "--run_mode", "single",

    "--model", MODEL_ID,
    "--model_revision", MODEL_REVISION,

    "--objective", OBJECTIVE,

    "--dataset", str(DATASET),
    "--directive_domain_registry_json",
    str(DIRECTIVE_DOMAIN_REGISTRY),

    "--split_mode", "family",

    "--val_families",
    "rodinia_pathfinder;machsuite_sort_radix",

    "--test_families",
    "serrano-kalman-filter",

    "--device_mode", "known",
    "--device_token_dropout", "0",

    "--resource_budget_mode", "random",
    "--random_budgets_per_case", "16",
    "--random_budget_min_frac", "0.10",
    "--min_feasible_candidates_per_budget", "3",
    "--candidate_pool_per_objective", "24",

    "--auto_frequency_fraction", "0",

    "--top_k", "1",
    "--min_supervised_sites", "2",
    "--min_site_coverage", "0.85",

    "--max_length", "7168",

    "--directive_loss_weighting", "uniform",

    "--batch_size", str(batch_size),
    "--grad_accum", str(grad_accum),

    "--num_workers", "2",
    "--group_by_length",
    "--gradient_checkpointing",

    "--lora_r", "8",
    "--lora_alpha", "16",
    "--lora_dropout", "0.05",

    "--selection_num_val_kernels", "0",
    "--selection_cases_per_kernel_device", "2",
    "--selection_eval_steps", "200",
    "--selection_candidate_batch_size", "4",

    "--loss_chunk_t", "256",
    "--seed", "123",
]

STAGE1_COMMON = COMMON_BASE + [
    "--disable_structural_memory",

    "--lr_lora", "5e-5",
    "--lr_embed", "5e-5",

    "--lr_xattn", "0",
    "--lr_gate", "0",
    "--lr_ff", "0",
    "--lr_gate_ff", "0",

    "--best_dir_name", "best_custom_stage1",
]

batch_size: 1
grad_accum: 8
effective batch: 8


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/632 [00:00<?, ?B/s]

Model: deepseek-ai/deepseek-coder-6.7b-base
Revision: ce2207a8bfef3ee92bd7dd4cc31c52cfa0046912


In [ ]:
import subprocess
from pathlib import Path

def checkpoint_is_complete(path: Path):
    required = [
        path / "trainer_state.json",
        path / "optimizer.pt",
        path / "scheduler.pt",
        path / "rng_state.pth",
        path / "adapter_config.json",
    ]

    has_adapter = (
        (path / "adapter_model.safetensors").is_file()
        or
        (path / "adapter_model.bin").is_file()
    )

    return (
        all(p.is_file() for p in required)
        and has_adapter
    )


all_checkpoints = sorted(
    FULL_DIR.glob("checkpoint-*"),
    key=lambda p: int(p.name.split("-")[-1]),
)

complete_checkpoints = [
    p for p in all_checkpoints
    if checkpoint_is_complete(p)
]

assert complete_checkpoints, (
    f"No complete resumable checkpoint found under {FULL_DIR}"
)

RESUME_CKPT = complete_checkpoints[-1]

print("All checkpoint directories:")
for p in all_checkpoints:
    print(
        " ",
        p.name,
        "COMPLETE" if checkpoint_is_complete(p) else "INCOMPLETE",
    )

print("\nResuming from:", RESUME_CKPT)

full_command = (
    STAGE1_COMMON
    + [
        "--split_json", str(SPLIT_JSON),

        "--output_dir", str(FULL_DIR),

        "--epochs", "3",
        "--max_steps", "-1",

        "--eval_steps", "200",
        "--save_steps", "200",
        "--selection_eval_steps", "200",

        "--resume_from_checkpoint",
        str(RESUME_CKPT),
    ]
)

FULL_LOG = (
    PERSIST_ROOT /
    f"stage1_{GOAL_TAG}_{RUN_ID}.log"
)

print("Launching resumed Stage-1 training")
print("Log:", FULL_LOG)
print("Output:", FULL_DIR)
print(" ".join(map(str, full_command)))

# IMPORTANT: append, don't destroy the existing training log.
with FULL_LOG.open(
    "a",
    encoding="utf-8",
    buffering=1,
) as log:

    log.write(
        "\n\n"
        + "=" * 100
        + f"\nRESUME FROM {RESUME_CKPT}\n"
        + "=" * 100
        + "\n"
    )
    log.flush()

    process = subprocess.Popen(
        full_command,
        cwd=REPO,
        env=ENV,

        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,

        text=True,
        bufsize=1,
    )

    assert process.stdout is not None

    for line in process.stdout:
        print(line, end="", flush=True)
        log.write(line)
        log.flush()

returncode = process.wait()

if returncode != 0:
    raise RuntimeError(
        f"Stage-1 resume failed: exit={returncode}. "
        f"See {FULL_LOG}"
    )

BEST_STAGE1 = FULL_DIR / "best_custom_stage1"
assert BEST_STAGE1.is_dir()

print("Stage-1 complete:", BEST_STAGE1)

Launching Stage-1 production training
Log: /content/drive/MyDrive/MailoHLS/Experiments/stage1_pareto_adp_s123_inference_fix.log
Output: /content/drive/MyDrive/MailoHLS/Experiments/stage1_pareto_adp_s123_inference_fix_full
/usr/bin/python3 -u LLM_branch/train/train_SFT_xattn_new.py --run_mode single --model deepseek-ai/deepseek-coder-6.7b-base --model_revision ce2207a8bfef3ee92bd7dd4cc31c52cfa0046912 --objective PARETO_ADP --dataset /content/MailoHLS/artifacts/llm/mailohls_sft.jsonl --directive_domain_registry_json /content/MailoHLS/artifacts/llm/directive_domain_registry.json --split_mode family --val_families rodinia_pathfinder;machsuite_sort_radix --test_families serrano-kalman-filter --device_mode known --device_token_dropout 0 --resource_budget_mode random --random_budgets_per_case 16 --random_budget_min_frac 0.10 --min_feasible_candidates_per_budget 3 --candidate_pool_per_objective 24 --auto_frequency_fraction 0 --top_k 1 --min_supervised_sites 2 --min_site_coverage 0.85 --max_len

In [ ]:
import json

BEST_STAGE1 = FULL_DIR / "best_custom_stage1"

required = [
    BEST_STAGE1 / "adapter_config.json",
    BEST_STAGE1 / "training_contract.json",
    BEST_STAGE1 / "tokenizer_config.json",
    BEST_STAGE1 / "best_selection_metrics.json",
]

for p in required:
    assert p.is_file(), p

assert (
    (BEST_STAGE1 / "adapter_model.safetensors").is_file()
    or
    (BEST_STAGE1 / "adapter_model.bin").is_file()
)

checkpoints = sorted(
    FULL_DIR.glob("checkpoint-*"),
    key=lambda p: int(p.name.split("-")[-1]),
)

print("Trainer checkpoints:")
for p in checkpoints:
    print("  ", p.name)

contract = json.loads(
    (BEST_STAGE1 / "training_contract.json")
    .read_text(encoding="utf-8")
)

assert contract["stage"] == "stage1"
assert contract["model"] == MODEL_ID
assert contract["model_revision"] == MODEL_REVISION

print(
    "Best metrics:",
    (
        BEST_STAGE1 /
        "best_selection_metrics.json"
    ).read_text()
)

In [ ]:
import subprocess
import json
import statistics
from collections import defaultdict

VAL_CASES = (
    FULL_DIR
    / "selected_debug"
    / f"val_selected_{GOAL_TAG}.jsonl"
)

EVAL_OUTPUT = (
    FULL_DIR /
    "stage1_val_predictions.jsonl"
)

assert VAL_CASES.is_file(), VAL_CASES

eval_command = [
    sys.executable, "-u",
    "LLM_branch/inference/eval_stage1_stage2_stage3.py",

    "--stage", "stage1",

    "--model", MODEL_ID,
    "--model_revision", MODEL_REVISION,

    "--adapter_dir", str(BEST_STAGE1),

    "--directive_domain_registry_json",
    str(DIRECTIVE_DOMAIN_REGISTRY),

    "--input_jsonl", str(VAL_CASES),
    "--output_jsonl", str(EVAL_OUTPUT),

    "--max_prompt_tokens", "7168",
    "--score_reduction", "mean",
]

process = subprocess.Popen(
    eval_command,
    cwd=REPO,
    env=ENV,

    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,

    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(line, end="", flush=True)

returncode = process.wait()

if returncode != 0:
    raise RuntimeError(
        f"Stage-1 inference failed: {returncode}"
    )

predictions = [
    json.loads(line)
    for line in EVAL_OUTPUT.read_text(
        encoding="utf-8"
    ).splitlines()
    if line.strip()
]

scored = [
    {
        **row,
        **row["candidates"][0],
    }
    for row in predictions
]

value_accuracy = statistics.mean(
    x["value_accuracy_over_expected"]
    for x in scored
)

schema_rate = statistics.mean(
    float(x["schema_compliant"])
    for x in scored
)

exact_rate = statistics.mean(
    float(x["exact_design_match"])
    for x in scored
)

print("Cases:", len(scored))
print("Slot accuracy:", value_accuracy)
print("Schema compliance:", schema_rate)
print("Exact-design accuracy:", exact_rate)

In [ ]:
from collections import Counter
import hashlib
import json
import torch

MEMORY_DIR = (
    REPO /
    "artifacts/stage2_inputs/"
    "memory_gnn_stage2_prod_s123"
)

manifest_path = MEMORY_DIR / "memory_manifest.json"
assert manifest_path.is_file()

manifest = json.loads(
    manifest_path.read_text(
        encoding="utf-8"
    )
)

assert manifest["schema"] == (
    "mailohls-memory-bank-manifest-v2"
)

assert manifest["embedding_mode"] == (
    "static_pre_npt"
)

assert manifest["checkpoint_tag"] == "val"
assert manifest["checkpoint_epoch"] == 15

SIDE_CAR = (
    REPO /
    "Checkpoints/"
    "gnn_stage2_prod_resource_s123/"
    "run1/"
    "val_model_state_dict.pth.json"
)

sidecar = json.loads(
    SIDE_CAR.read_text(encoding="utf-8")
)

assert (
    manifest["gnn_checkpoint_sha256"]
    == sidecar["checkpoint_sha256"]
)

assert (
    manifest["gnn_contract_sha256"]
    == sidecar["contract_sha256"]
)

memory_files = sorted(
    MEMORY_DIR.glob("*.memory.pt")
)

assert len(memory_files) == 55, len(memory_files)

active_counts = []
category_counts = Counter()

for path in memory_files:

    pack = torch.load(
        path,
        map_location="cpu",
        weights_only=False,
    )

    assert pack["embedding_mode"] == "static_pre_npt"

    assert bool(
        pack["disable_pragma_injection"]
    )

    emb = pack["node_embs"]
    mask = pack["node_embs_mask"]
    cats = pack["slot_cats"]

    assert emb.shape == (64, 64)
    assert mask.shape == (64,)
    assert cats.shape == (64,)

    assert torch.isfinite(emb).all()

    n_active = int(mask.sum())
    assert n_active > 0

    active_counts.append(n_active)

    for cat in cats[mask].tolist():
        assert cat in (1, 2)
        category_counts[int(cat)] += 1

    labels = pack["labels"]

    for slot in mask.nonzero(
        as_tuple=False
    ).view(-1).tolist():

        assert labels[slot] == slot + 1

print("Memory bank OK")
print("kernels:", len(memory_files))
print(
    "active slots:",
    min(active_counts),
    sum(active_counts) / len(active_counts),
    max(active_counts),
)
print(
    "categories:",
    category_counts,
)
print(
    "manifest SHA:",
    hashlib.sha256(
        manifest_path.read_bytes()
    ).hexdigest()
)

In [ ]:
STAGE2_SMOKE_DIR = (
    PERSIST_ROOT /
    f"stage2_{GOAL_TAG}_{RUN_ID}_smoke"
)
INITIAL_XATTN_STATE_REFERENCE = (
    PERSIST_ROOT /
    "initial_harp_state_post_sa_pre_mlp_s123.json"
)

assert not STAGE2_SMOKE_DIR.exists()

stage2_command = COMMON_BASE + [
    "--split_json", str(SPLIT_JSON),

    "--output_dir",
    str(STAGE2_SMOKE_DIR),

    "--init_adapter_dir",
    str(BEST_STAGE1),

    "--memory_dir",
    str(MEMORY_DIR),

    "--require_pragma_free_memory",

    "--initial_state_reference",
    str(INITIAL_XATTN_STATE_REFERENCE),

    "--best_dir_name",
    "best_custom_stage2",

    "--lr_lora", "0",
    "--lr_embed", "0",

    "--lr_xattn", "1e-4",
    "--lr_gate", "2e-4",

    "--lr_ff", "0",
    "--lr_gate_ff", "0",

    "--max_slots", "64",
    "--every_n_layers", "8",
    "--xattn_heads", "4",
    "--xattn_dim_head", "64",

    "--epochs", "1",
    "--max_steps", "20",

    "--eval_steps", "20",
    "--save_steps", "20",

    "--selection_eval_steps", "20",
    "--selection_num_val_kernels", "1",
    "--selection_cases_per_kernel_device", "1",
]

STAGE2_LOG = (
    PERSIST_ROOT /
    f"stage2_{GOAL_TAG}_{RUN_ID}_smoke.log"
)

with STAGE2_LOG.open(
    "w",
    encoding="utf-8",
    buffering=1,
) as log:

    process = subprocess.Popen(
        stage2_command,
        cwd=REPO,
        env=ENV,

        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,

        text=True,
        bufsize=1,
    )

    for line in process.stdout:
        print(line, end="", flush=True)
        log.write(line)
        log.flush()

returncode = process.wait()

if returncode != 0:
    raise RuntimeError(
        f"Stage-2 smoke failed: {returncode}"
    )

BEST_STAGE2 = (
    STAGE2_SMOKE_DIR /
    "best_custom_stage2"
)

assert BEST_STAGE2.is_dir()
assert (
    BEST_STAGE2 /
    "harp_xattn.pt"
).is_file()

stage2_contract = json.loads(
    (
        BEST_STAGE2 /
        "training_contract.json"
    ).read_text(encoding="utf-8")
)

assert stage2_contract["stage"] == "stage2"

struct = stage2_contract["structural"]

assert struct["mem_dim"] == 64
assert struct["max_slots"] == 64
assert struct["every_n_layers"] == 8
assert struct["xattn_heads"] == 4
assert struct["xattn_dim_head"] == 64

print("Stage-2 smoke passed.")
print(json.dumps(struct, indent=2))

In [ ]:
stage2_cases = []

with VAL_CASES.open(
    "r",
    encoding="utf-8",
) as handle:
    for line in handle:
        if line.strip():
            stage2_cases.append(line)
        if len(stage2_cases) >= 4:
            break

STAGE2_CASES = (
    STAGE2_SMOKE_DIR /
    "stage2_eval_cases.jsonl"
)

STAGE2_CASES.write_text(
    "".join(stage2_cases),
    encoding="utf-8",
)

STAGE2_EVAL_OUTPUT = (
    STAGE2_SMOKE_DIR /
    "stage2_val_predictions.jsonl"
)

stage2_eval_command = [
    sys.executable, "-u",
    "LLM_branch/inference/"
    "eval_stage1_stage2_stage3.py",

    "--stage", "stage2",

    "--model", MODEL_ID,
    "--model_revision", MODEL_REVISION,

    "--adapter_dir",
    str(BEST_STAGE2),

    "--memory_dir",
    str(MEMORY_DIR),

    "--directive_domain_registry_json",
    str(DIRECTIVE_DOMAIN_REGISTRY),

    "--input_jsonl",
    str(STAGE2_CASES),

    "--output_jsonl",
    str(STAGE2_EVAL_OUTPUT),

    "--max_prompt_tokens", "7168",

    "--score_reduction", "mean",
]

process = subprocess.Popen(
    stage2_eval_command,
    cwd=REPO,
    env=ENV,

    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,

    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(line, end="", flush=True)

returncode = process.wait()

if returncode != 0:
    raise RuntimeError(
        f"Stage-2 inference failed: {returncode}"
    )

print(
    STAGE2_EVAL_OUTPUT.read_text(
        encoding="utf-8"
    )
)